# 01 Metrics Rus O

Computes the full metric set for the RUS-O subcorpus from the running text.

**Inputs:** `kid_lit_100_ru.csv`  
**Outputs:** `all_metrics_ru.csv`, `summary_table_ru.csv`

> **Not re-runnable from this repository.** This notebook reads the running text of the 100 books, which is under copyright and is not distributed. It is included as a record of how the released metrics were produced.


In [ ]:
!pip install razdel pyphen --quiet
!python -m spacy download ru_core_news_sm --quiet
print("✅ Готово — перезапустите среду выполнения!")


# Импорты и константы

In [ ]:
import csv
import math
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from razdel import tokenize, sentenize
import pyphen
import spacy

sns.set_theme(style='whitegrid', palette='muted')
# ACL-ready figure styling: large, legible text for single-column print
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 14,
    'axes.titlesize': 15, 'axes.titleweight': 'bold', 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'legend.fontsize': 12, 'figure.titlesize': 17, 'figure.titleweight': 'bold',
    'savefig.dpi': 300, 'figure.dpi': 110, 'savefig.bbox': 'tight',
})

# spacy — для синтаксиса, грамматики, POS-тегов (оставляем)
dic = pyphen.Pyphen(lang='ru_RU')
nlp = spacy.load("ru_core_news_sm")

MEANINGFUL_POS = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}

PUNCT_MARKS = {
    'dot':         '.',
    'comma':       ',',
    'excl':        '!',
    'quest':       '?',
    'ellipsis':    '…',
    'semicolon':   ';',
    'colon':       ':',
    'dash':        '—',
    'hyphen':      '-',
    'quote_open':  '«',
    'quote_close': '»',
}

RU_TOP1000 = {
    'и','в','не','он','на','я','что','тот','быть','с','а','весь','это',
    'как','она','по','но','они','к','из','у','за','так','его','то',
    'все','один','свой','вот','от','же','мы','при','о','об','её','уже',
    'бы','мой','до','вы','мне','если','со','есть','ли','ну','где','ещё',
    'там','тебе','никто','который','также','наш','ты','эти','был',
    'или','себя','нас','нет','да','этот','даже','тем','ним','через',
    'для','между','два','три','четыре','пять','шесть','семь','восемь',
    'девять','десять','такой','очень','здесь','сейчас','хотеть','мочь',
    'сказать','говорить','знать','видеть','стать','идти','время','год',
    'человек','день','рука','место','лицо','дело','жизнь','слово','дом',
    'друг','думать','спросить','ответить','понять','пойти','взять',
    'стоять','сидеть','пусть','потом','потому','когда','сначала','раз',
    'конечно','однако','хорошо','новый','большой','маленький','старый',
    'молодой','первый','последний','другой','нужный','самый','разный',
    'каждый','только','тоже','всё','просто','теперь','лишь',
    'именно','совсем','более','менее','почти','вдруг','вместе','всегда',
    'часто','никогда','снова','сразу','больше','меньше','лучше','хуже',
    'мать','отец','сын','дочь','ребёнок','дети','семья','брат','сестра',
    'мама','папа','бабушка','дедушка','голова','глаз','нога','нос','рот',
    'вода','земля','небо','солнце','свет','ночь','утро','вечер','час',
    'минута','школа','работа','город','страна','мир','путь','конец',
    'начало','часть','вид','сторона','сила','случай','вопрос','ответ',
}

print("✅ OK!")


# Google Drive и пути

In [ ]:
# --- Paths ---------------------------------------------------------------
# BASE_PATH must point at a directory holding this notebook's input files.
# Set the KIDLIT_BASE environment variable, or edit the fallback below.
# In Google Colab: mount Drive first, then set KIDLIT_BASE to the folder there.
import os
BASE_PATH = os.environ.get("KIDLIT_BASE", "../data/")
CSV_FILE    = BASE_PATH + 'kid_lit_100_ru.csv'
GROUP_LABEL = 'Russian literature (original)'

import os
os.makedirs(GRAPH_PATH, exist_ok=True)

def save(name):
    plt.savefig(GRAPH_PATH + name + '.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ {name}.png")


# Загрузка данных

In [ ]:
USEFUL_COLS = [
    'id','title','original_title','author','author_gender','year',
    'language','country_of_origin','publisher','age_marker','pages',
    'illustrator','УДК','awards','is_translation','translator',
    'description','text','expert',
]

_available = pd.read_csv(CSV_FILE, sep=';', encoding='utf-8-sig', nrows=0).columns.tolist()
_cols = [c for c in USEFUL_COLS if c in _available]
df = pd.read_csv(CSV_FILE, sep=';', encoding='utf-8-sig', usecols=_cols)
df['author_gender'] = df['author_gender'].str.strip()
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['period'] = df['year'].apply(
    lambda y: 'before 2020' if pd.notna(y) and int(y) < 2020 else '2020 and later'
)
df['gender_clean'] = df['author_gender'].apply(
    lambda g: 'collective' if str(g).strip() == 'жен/муж' else str(g).strip()
)

df_texts = df[df['text'].notna()].copy().reset_index(drop=True)

print(f"📚 Всего книг:       {len(df)}")
print(f"📝 Книг с текстом:  {len(df_texts)}")
print(f"⚠️  Без текста:      {df['text'].isna().sum()}")
print(f"\nПол автора:  {df['gender_clean'].value_counts().to_dict()}")
print(f"Период:      {df['period'].value_counts().to_dict()}")

GENDER_ORDER  = [g for g in ['female','male','collective'] if g in df_texts['gender_clean'].unique()]
PERIOD_ORDER  = ['before 2020','2020 and later']
GENDER_COLORS = {'female':'#f4a7b9','male':'#7eb8d4','collective':'#a8d8a8'}
PERIOD_COLORS = {'before 2020':'#c0c0c0','2020 and later':'#7eb8d4'}


# Вспомогательные функции

In [ ]:
def get_words(txt):
    return [t.text for t in tokenize(str(txt))
            if re.match(r'^[а-яёА-ЯЁ]+$', t.text)]

def get_sentences(txt):
    return [s.text for s in sentenize(str(txt))]

def count_syllables(word):
    s = dic.inserted(word.lower())
    if not s:
        return max(1, sum(1 for c in word.lower() if c in 'аеёиоуыэюя'))
    return s.count('-') + 1

def get_meaningful_words(txt, lemmatize=True):
    """Content words (NOUN/VERB/ADJ/ADV/PROPN) as lowercased lemmas."""
    if not txt or pd.isna(txt):
        return []
    doc = nlp(str(txt)[:100000])
    result = []
    for token in doc:
        if token.pos_ not in MEANINGFUL_POS:
            continue
        if token.is_stop or token.is_punct or not token.is_alpha or len(token.text) <= 2:
            continue
        result.append(token.lemma_.lower() if lemmatize else token.text.lower())
    return result

def get_tree_depth(token):
    children = list(token.children)
    return 1 if not children else 1 + max(get_tree_depth(c) for c in children)

def sr(num, denom):
    return round(num / denom, 4) if denom > 0 else 0

print("✅ Функции готовы")


# Вычисление метрик

In [ ]:
print("⏳ Считаем метрики (5–10 минут)...")

all_results = []

for _, row in df_texts.iterrows():
    txt     = str(row['text'])
    desc    = str(row['description']) if pd.notna(row['description']) else ''
    book_id = row['id']

    words   = get_words(txt)
    sents   = get_sentences(txt)
    wlens   = [len(w) for w in words]
    slens_w = [len(get_words(s)) for s in sents]
    slens_c = [len(s) for s in sents]
    sylls   = [count_syllables(w) for w in words]
    freq    = Counter(words)

    # A. Длина
    n_chars       = len(txt)
    n_chars_no_sp = len(txt.replace(' ', ''))
    n_words       = len(words)
    n_unique      = len(set(words))
    n_sents       = len(sents)
    n_syllables   = sum(sylls)

    # B. Слово
    avg_word_len  = round(np.mean(wlens), 3)   if wlens else 0
    med_word_len  = round(np.median(wlens), 3) if wlens else 0
    avg_syllables = round(np.mean(sylls), 3)   if sylls else 0
    n_mono        = sum(1 for s in sylls if s == 1)
    n_poly        = sum(1 for s in sylls if s >= 3)
    fog           = round(0.4 * (n_words / n_sents + n_poly / n_words * 100), 3) \
                    if n_sents > 0 and n_words > 0 else 0

    # C. Предложение
    avg_sent_w = round(np.mean(slens_w), 3)   if slens_w else 0
    avg_sent_c = round(np.mean(slens_c), 3)   if slens_c else 0
    med_sent   = round(np.median(slens_w), 3) if slens_w else 0
    min_sent   = min(slens_w) if slens_w else 0
    max_sent   = max(slens_w) if slens_w else 0
    n_short    = sum(1 for l in slens_w if l <= 5)
    n_long     = sum(1 for l in slens_w if l >= 20)

    # D. Lexical diversity (all content words)
    words_lex  = get_meaningful_words(txt)
    freq_lex   = Counter(words_lex)
    n_lex      = len(words_lex)
    n_uniq_lex = len(freq_lex)

    ttr      = round(n_uniq_lex / n_lex, 4) if n_lex > 0 else 0
    seg_size = 1000
    ttrs     = [len(set(words_lex[i:i+seg_size])) / len(words_lex[i:i+seg_size])
                for i in range(0, n_lex, seg_size)
                if len(words_lex[i:i+seg_size]) >= 100]
    sttr     = round(np.mean(ttrs), 4) if ttrs else ttr

    hapax       = sum(1 for c in freq_lex.values() if c == 1)
    hapax_ratio = round(hapax / n_lex, 4)       if n_lex > 0 else 0
    hapax_dis   = round(sum(1 for c in freq_lex.values() if c == 2) / n_lex, 4) \
                  if n_lex > 0 else 0

    fv      = np.array(list(freq_lex.values())) if freq_lex else np.array([1])
    m1, m2  = len(fv), np.sum(fv ** 2)
    yule_k  = round(10000 * (m2 - m1) / (m1 ** 2), 4) if m1 > 1 else 0
    simp_d  = round(np.sum(fv * (fv - 1)) / (m1 * (m1 - 1)), 4) if m1 > 1 else 0
    probs   = fv / fv.sum()
    shannon = round(-np.sum(probs * np.log2(probs + 1e-10)), 4)
    top1000 = sr(sum(1 for w in words if w.lower() in RU_TOP1000), n_words)

    # E. Читаемость
    flesch  = round(206.835 - 1.015 * avg_sent_w - 84.6 * avg_syllables, 3)
    fk      = round(0.39 * avg_sent_w + 11.8 * avg_syllables - 15.59, 3)
    smog    = round(3 + math.sqrt(n_poly * 30 / n_sents), 3) if n_sents >= 30 else 0
    L       = n_chars_no_sp / n_words * 100 if n_words > 0 else 0
    S_      = n_sents / n_words * 100 if n_words > 0 else 0
    coleman = round(0.0588 * L - 0.296 * S_ - 15.8, 3)
    ari     = round(4.71 * (n_chars_no_sp / n_words) + 0.5 * avg_sent_w - 21.43, 3) \
              if n_words > 0 else 0

    # F. Синтаксис
    doc = nlp(txt[:100000])
    depths, deps_pt, coord_f = [], [], []
    for sent in doc.sents:
        toks  = list(sent)
        roots = [t for t in toks if t.dep_ == 'ROOT']
        if roots:
            depths.append(get_tree_depth(roots[0]))
        deps_pt.append(np.mean([len(list(t.children)) for t in toks]))
        coord_f.append(int(any(t.dep_ in ('conj','cc') for t in toks)))

    avg_depth   = round(np.mean(depths), 3)   if depths  else 0
    max_depth   = max(depths)                  if depths  else 0
    avg_deps    = round(np.mean(deps_pt), 3)   if deps_pt else 0
    coord_ratio = round(np.mean(coord_f), 3)   if coord_f else 0

    # G. Грамматика
    all_toks = [t for t in doc if t.is_alpha]
    n_all    = len(all_toks)
    verbs    = [t for t in all_toks if t.pos_ == 'VERB']
    n_v      = len(verbs)
    nouns    = [t for t in all_toks if t.pos_ in ('NOUN','PROPN')]
    prons    = [t for t in all_toks if t.pos_ == 'PRON']

    act_v   = sr(sum(1 for t in verbs if t.morph.get('Voice') == ['Act']),  n_v)
    pass_v  = sr(sum(1 for t in verbs if t.morph.get('Voice') == ['Pass']), n_v)
    past_v  = sr(sum(1 for t in verbs if t.morph.get('Tense') == ['Past']), n_v)
    pres_v  = sr(sum(1 for t in verbs if t.morph.get('Tense') == ['Pres']), n_v)
    fut_v   = sr(sum(1 for t in verbs if t.morph.get('Tense') == ['Fut']),  n_v)
    imp_v   = sr(sum(1 for t in verbs if t.morph.get('Mood') == ['Imp']),   n_v)
    pers_pr = sr(sum(1 for t in prons  if t.morph.get('PronType') == ['Prs']), n_all)
    anim_n  = sr(sum(1 for t in nouns  if t.morph.get('Animacy') == ['Anim']), len(nouns))
    pos_counts = Counter(t.pos_ for t in all_toks)
    pos_ratio  = {f'pos_{p.lower()}_ratio': sr(c, n_all) for p, c in pos_counts.items()}

    # H. Пунктуация
    punct_res = {}
    all_p = [c for c in txt if c in ''.join(PUNCT_MARKS.values())]
    for name, char in PUNCT_MARKS.items():
        punct_res[f'punct_{name}_ratio'] = sr(txt.count(char), n_words)
    punct_res['punct_diversity']  = round(len(set(all_p)) / len(PUNCT_MARKS), 3)
    punct_res['quest_sent_ratio'] = sr(sum(1 for s in sents if '?' in s), n_sents)
    punct_res['excl_sent_ratio']  = sr(sum(1 for s in sents if '!' in s), n_sents)

    # J. Аннотация
    dw  = [t.text.lower() for t in tokenize(desc) if re.match(r'^[а-яёА-ЯЁ]+$', t.text)]
    dws = set(dw)
    tws = set(w.lower() for w in words)
    jac = round(len(dws & tws) / len(dws | tws), 4) if (dws | tws) else 0
    ovl = round(len(dws & tws) / len(dws),       4) if dws else 0
    desc_m = {
        'desc_n_words': len(dw),
        'desc_n_sents': len(get_sentences(desc)),
        'desc_ttr':     round(len(dws)/len(dw), 4) if dw else 0,
        'desc_jaccard': jac,
        'desc_overlap': ovl,
    }

    record = {
        'id': row['id'], 'title': row['title'], 'author': row['author'],
        'gender_clean': row['gender_clean'], 'year': row['year'],
        'period': row['period'], 'publisher': row['publisher'], 'pages': row['pages'],
        # Длина
        'n_chars': n_chars, 'n_chars_no_spaces': n_chars_no_sp,
        'n_words': n_words, 'n_unique_words': n_unique,
        'n_sentences': n_sents, 'n_syllables': n_syllables,
        # Lexical counts (all content words)
        'n_lex_words': n_lex, 'n_lex_unique': n_uniq_lex,
        # Слово
        'avg_word_len': avg_word_len, 'med_word_len': med_word_len,
        'avg_syllables': avg_syllables,
        'n_monosyllabic': n_mono, 'n_polysyllabic': n_poly, 'fog_index': fog,
        # Предложение
        'avg_sent_words': avg_sent_w, 'avg_sent_chars': avg_sent_c,
        'med_sent_len': med_sent, 'min_sent_len': min_sent,
        'max_sent_len': max_sent, 'n_short_sents': n_short, 'n_long_sents': n_long,
        # Lexical diversity (all content words)
        'ttr': ttr, 'sttr': sttr,
        'hapax_count': hapax, 'hapax_ratio': hapax_ratio, 'hapax_dis_ratio': hapax_dis,
        'yule_k': yule_k, 'simpson_d': simp_d, 'shannon_h': shannon, 'top1000_ratio': top1000,
        # Читаемость
        'flesch_ru': flesch, 'flesch_kincaid': fk,
        'smog': smog, 'coleman_liau': coleman, 'ari': ari,
        # Синтаксис
        'avg_tree_depth': avg_depth, 'max_tree_depth': max_depth,
        'avg_dependents': avg_deps, 'coord_sent_ratio': coord_ratio,
        # Грамматика
        'active_voice_ratio': act_v, 'passive_voice_ratio': pass_v,
        'past_tense_ratio': past_v, 'present_tense_ratio': pres_v,
        'future_tense_ratio': fut_v, 'imperative_ratio': imp_v,
        'personal_pron_ratio': pers_pr, 'animate_noun_ratio': anim_n,
        **punct_res, **desc_m,
    }
    record.update(pos_ratio)
    all_results.append(record)
    print(f"  ✓ {row['title'][:50]}")

mdf = pd.DataFrame(all_results)
mdf.to_csv(GRAPH_PATH + 'all_metrics_ru.csv', sep='\t', index=False, quoting=csv.QUOTE_ALL)
print(f"\n✅ Строк: {len(mdf)}, Колонок: {len(mdf.columns)}")
mdf[['title','n_words','n_lex_words','n_sentences','ttr','flesch_ru']].head(8)


# Сводная таблица

In [ ]:
print("\n" + "═"*65)
print(f"  SUMMARY TABLE — {GROUP_LABEL.upper()}")
print("═"*65)

summary_groups = {
    'Characters and words': {
        'Total characters':            'n_chars',
        'Characters (no spaces)':     'n_chars_no_spaces',
        'Total words':                'n_words',
        'Unique words':           'n_unique_words',
        'Avg. word length (chars)': 'avg_word_len',
    },
    'Syllables': {
        'Total syllables':              'n_syllables',
        'Avg. syllables/word':        'avg_syllables',
        'Monosyllabic words':          'n_monosyllabic',
        'Polysyllabic (3+ syl.)':   'n_polysyllabic',
    },
    'Sentences': {
        'Total sentences':         'n_sentences',
        'Avg. sent. length (words)':   'avg_sent_words',
        'Median sentence length':      'med_sent_len',
        'Min sentence length':         'min_sent_len',
        'Max sentence length':        'max_sent_len',
        'Short sentences (≤5 w.)':   'n_short_sents',
        'Long sentences (≥20 w.)':   'n_long_sents',
    },
    'Lexical diversity': {
        'Content words':          'n_lex_words',
        'Unique lexemes':         'n_lex_unique',
        'TTR':                       'ttr',
        'STTR':                      'sttr',
        'Hapax ratio':              'hapax_ratio',
        'Yule K index':              'yule_k',
        'Shannon entropy':          'shannon_h',
        'Top-1000 word ratio':        'top1000_ratio',
    },
    'Readability': {
        'Flesch (RU)':             'flesch_ru',
        'Flesch-Kincaid':            'flesch_kincaid',
        'Coleman-Liau':              'coleman_liau',
        'ARI':                       'ari',
        'FOG index':                       'fog_index',
    },
    'Syntax': {
        'Avg. tree depth':        'avg_tree_depth',
        'Max tree depth':      'max_tree_depth',
        'Avg. dependents per token':    'avg_dependents',
        'Coord. clauses ratio': 'coord_sent_ratio',
    },
    'Grammar': {
        'Active voice ratio':    'active_voice_ratio',
        'Passive voice ratio':     'passive_voice_ratio',
        'Past tense ratio':           'past_tense_ratio',
        'Present tense ratio':           'present_tense_ratio',
        'Future tense ratio':             'future_tense_ratio',
        'Personal pronoun ratio': 'personal_pron_ratio',
        'Animate noun ratio':      'animate_noun_ratio',
    },
}

rows = []
for group, metrics in summary_groups.items():
    for label, col in metrics.items():
        if col in mdf.columns:
            rows.append({
                'Group': group, 'Metric': label,
                'Mean':   round(mdf[col].mean(), 3),
                'Median':   round(mdf[col].median(), 3),
                'Min':       round(mdf[col].min(), 3),
                'Max':       round(mdf[col].max(), 3),
                'Std':       round(mdf[col].std(), 3),
            })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(GRAPH_PATH + 'summary_table_ru.csv', sep='\t', index=False, encoding='utf-8-sig')

for group in summary_groups:
    sub = summary_df[summary_df['Group'] == group]
    print(f"\n▶ {group}")
    print(sub[['Metric','Mean','Median','Min','Max','Std']].to_string(index=False))

print("\n✅ Saved → summary_table_ru.csv")


# Распределение длин — гистограммы + boxplot + bar

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle(f'Text length distribution — {GROUP_LABEL}')
panels = [
    (axes[0], 'n_chars',     'Characters', '#4878a8'),
    (axes[1], 'n_words',     'Words',      '#d1755b'),
    (axes[2], 'n_sentences', 'Sentences',  '#5a9e6f'),
    (axes[3], 'n_syllables', 'Syllables',  '#8d72b5'),
]
for ax, col, label, color in panels:
    ax.hist(mdf[col], bins=10, color=color, edgecolor='white', alpha=0.88)
    ax.axvline(mdf[col].mean(),   color='black', ls='--', lw=1.6)
    ax.axvline(mdf[col].median(), color='red',   ls=':',  lw=1.6)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('Number of books')
plt.tight_layout()
save('plot_length_distribution')


# Распределение всех ключевых метрик


In [ ]:
dist_metrics = [
    ('avg_word_len',        'Avg. word length'),
    ('avg_syllables',       'Avg. syllables/word'),
    ('avg_sent_words',      'Avg. sentence length'),
    ('ttr',                 'TTR'),
    ('sttr',                'STTR'),
    ('hapax_ratio',         'Hapax ratio'),
    ('yule_k',              'Yule K index'),
    ('shannon_h',           'Shannon entropy'),
    ('flesch_ru',           'Flesch (RU)'),
    ('fog_index',           'FOG index'),
    ('ari',                 'ARI'),
    ('coleman_liau',        'Coleman-Liau'),
    ('avg_tree_depth',      'Syntactic tree depth'),
    ('avg_dependents',      'Dependents per token'),
    ('active_voice_ratio',  'Active voice'),
    ('personal_pron_ratio', 'Personal pronouns'),
    ('top1000_ratio',       'Top-1000 word ratio'),
    ('punct_diversity',     'Punctuation diversity'),
]
dist_metrics = [(c, l) for c, l in dist_metrics if c in mdf.columns]

n_cols = 4
n_rows = math.ceil(len(dist_metrics) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4.5, n_rows*3.6))
fig.suptitle(f'Key metric distributions — {GROUP_LABEL}')

for ax, (col, label) in zip(axes.flat, dist_metrics):
    ax.hist(mdf[col], bins=8, color='#4878a8', edgecolor='white', alpha=0.88)
    ax.axvline(mdf[col].mean(),   color='black', ls='--', lw=1.4)
    ax.axvline(mdf[col].median(), color='red',   ls=':',  lw=1.4)
    ax.set_title(label, fontsize=13)
    ax.set_ylabel('N books', fontsize=11)

for j in range(len(dist_metrics), len(axes.flat)):
    axes.flat[j].set_visible(False)

plt.tight_layout()
save('plot_all_distributions')


# Корреляционная матрица


In [ ]:
corr_cols = [
    'n_words','n_unique_words','n_sentences','avg_word_len',
    'avg_syllables','avg_sent_words','ttr','sttr','hapax_ratio',
    'yule_k','shannon_h','flesch_ru','fog_index','ari','coleman_liau',
    'avg_tree_depth','avg_dependents','coord_sent_ratio',
    'active_voice_ratio','personal_pron_ratio','animate_noun_ratio','top1000_ratio',
]
corr_cols = [c for c in corr_cols if c in mdf.columns]
corr = mdf[corr_cols].corr()

plt.figure(figsize=(16, 14))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
            annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, annot_kws={'size': 9})
plt.title(f'Metric correlation matrix — {GROUP_LABEL}')
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
save('plot_correlation')


# Синтаксис

In [ ]:
syntax_metrics = [
    ('avg_tree_depth',   'Avg. syntactic tree depth'),
    ('max_tree_depth',   'Max syntactic tree depth'),
    ('avg_dependents',   'Avg. dependents per token'),
    ('coord_sent_ratio', 'Coord. clauses ratio'),
    ('avg_sent_words',   'Avg. sentence length (words)'),
    ('n_long_sents',     'Long sentences (≥20 words)'),
]
syntax_metrics = [(c, l) for c, l in syntax_metrics if c in mdf.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle(f'Syntactic metrics — {GROUP_LABEL}')

for ax, (col, label) in zip(axes.flat, syntax_metrics):
    ax.hist(mdf[col], bins=10, color='#5a8fb5', edgecolor='white', alpha=0.88)
    ax.axvline(mdf[col].mean(),   color='black', ls='--', lw=1.4)
    ax.axvline(mdf[col].median(), color='red',   ls=':',  lw=1.4)
    ax.set_title(label, fontsize=13)
    ax.set_ylabel('N books', fontsize=11)

plt.tight_layout()
save('plot_syntax')


# Frequent content words


In [ ]:
all_lex = []
for _, row in df_texts.iterrows():
    all_lex.extend(get_meaningful_words(str(row['text'])))

top20 = Counter(all_lex).most_common(20)
words_t, counts_t = zip(*top20)

plt.figure(figsize=(14, 6))
plt.bar(words_t, counts_t, color='#4878a8', edgecolor='white')
plt.title(f'Top-20 content words — {GROUP_LABEL}')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Frequency')
plt.tight_layout()
save('plot_top20_words')


# Итог


In [ ]:
print("\n" + "═"*60)
print(f"  SUMMARY — {GROUP_LABEL.upper()}")
print("═"*60)
print(f"  Books analysed:    {len(mdf)}")
print(f"  Metrics computed:  {len(mdf.columns)}")
print(f"\n  Data files:")
for f in ['all_metrics_ru.csv', 'summary_table_ru.csv']:
    print(f"     {f}")
print(f"\n  Figures:")
for p in ['plot_length_distribution', 'plot_all_distributions',
          'plot_correlation', 'plot_syntax', 'plot_top20_words']:
    print(f"     {p}.png")
